# Video Generation on Colab — LTX-Video 2.3 & WAN 2.2

Generate videos from text prompts (or an image) using open-source diffusion models.
No Ollama — these models run directly via HuggingFace `diffusers`.

**Supported models:**

| Model key | HF model | Type | Min VRAM | Notes |
|---|---|---|---|---|
| `ltx-2.3` | Lightricks/LTX-Video | Text→Video | 16 GB (T4) | Fast, good quality |
| `ltx-2.3-i2v` | Lightricks/LTX-Video | Image→Video | 16 GB (T4) | Animate a starting image |
| `wan-2.1-1.3b` | Wan-AI/Wan2.1-T2V-1.3B-Diffusers | Text→Video | 16 GB (T4) | Smallest WAN, fits T4 |
| `wan-2.2-a14b` | Wan-AI/Wan2.2-T2V-A14B-Diffusers | Text→Video | 40 GB (A100) | MoE, 14B active / 27B total |

**Before running:**
- Runtime → Change runtime type → **GPU** (T4 for LTX or WAN 1.3B; A100 40 GB+ for WAN 2.2 A14B)
- Run cells top to bottom
- Output video downloads directly from the last cell

---

## 1. Check GPU

Confirm VRAM so you know which model to pick.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Pick your model & settings

Set `MODEL` to one of:

- `"ltx-2.3"` — LTX-Video 2.3 text-to-video (T4+)
- `"ltx-2.3-i2v"` — LTX-Video 2.3 image-to-video (T4+, set `IMAGE_PATH` below)
- `"wan-2.1-1.3b"` — WAN 2.1 T2V 1.3B — small model, fits T4
- `"wan-2.2-a14b"` — WAN 2.2 T2V A14B — best quality, needs A100 40 GB+

Resolution presets are chosen automatically based on your GPU; you can override below.

In [ ]:
# ── MODEL ──────────────────────────────────────────────────────────────────
MODEL = "ltx-2.3"   # ltx-2.3 | ltx-2.3-i2v | wan-2.1-1.3b | wan-2.2-a14b

# ── PROMPT ─────────────────────────────────────────────────────────────────
PROMPT = (
    "A serene mountain lake at golden hour, gentle ripples on the water surface, "
    "pine trees reflected in the lake, cinematic, 4K, photorealistic"
)
NEGATIVE_PROMPT = (
    "worst quality, inconsistent motion, blurry, jittery, distorted, "
    "watermark, text, logo, low resolution"
)

# ── FOR IMAGE-TO-VIDEO ONLY ─────────────────────────────────────────────────
# Upload your image to Colab (Files panel on the left) and set the path here.
IMAGE_PATH = "/content/my_image.png"   # only used when MODEL == "ltx-2.3-i2v"

# ── RESOLUTION & LENGTH (override or leave as None for auto) ───────────────
WIDTH  = None   # e.g. 768 — set None to use the default for your GPU/model
HEIGHT = None   # e.g. 512
NUM_FRAMES = None   # e.g. 121 (LTX), 81 (WAN)  ← more frames = longer video, more VRAM

# ── QUALITY / SPEED ────────────────────────────────────────────────────────
NUM_STEPS = 50      # inference steps — 30 is fast, 50 is good, 75 is slow but best
GUIDANCE_SCALE = 3.5
SEED = 42
FPS = 24            # output frames per second

# ── VRAM SAVING ────────────────────────────────────────────────────────────
# True  = slower generation but uses ~40% less VRAM.  Recommended for T4.
# False = faster, needs more VRAM.  Use on L4 / A100.
CPU_OFFLOAD = True

# ── OUTPUT ─────────────────────────────────────────────────────────────────
OUTPUT_PATH = "/content/output_video.mp4"

print(f"Model : {MODEL}")
print(f"Prompt: {PROMPT[:80]}...")

## 3. Install dependencies

One-time install (~3-5 minutes). Re-running is fast if already cached.

In [ ]:
!pip install -q --upgrade \
    "diffusers>=0.32.0" \
    transformers \
    accelerate \
    sentencepiece \
    "imageio[ffmpeg]" \
    imageio-ffmpeg \
    "Pillow>=10.2,<11"

# Verify key versions
import importlib
for pkg in ["diffusers", "transformers", "accelerate", "torch", "PIL"]:
    mod = importlib.import_module(pkg)
    ver = getattr(mod, "__version__", None) or getattr(mod, "PILLOW_VERSION", "?")
    print(f"{pkg}: {ver}")

## 4. Compute default resolution / frame count for your GPU

If you left `WIDTH`, `HEIGHT`, or `NUM_FRAMES` as `None` above, this cell picks sensible defaults that fit your available VRAM.

In [ ]:
import subprocess, re

def get_vram_gb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True
        )
        return int(out.strip().split("\n")[0]) / 1024  # MiB → GB
    except Exception:
        return 0

vram = get_vram_gb()
print(f"Detected VRAM: {vram:.1f} GB")

# --- LTX-Video defaults ---
_ltx_defaults = {
    # (width, height, num_frames)
    80:  (1280, 720,  161),  # H100 / A100 80GB
    40:  (1024, 576,  161),  # A100 40GB
    24:  (768,  512,  121),  # L4
    16:  (512,  320,   65),  # T4  (CPU offload recommended)
}
# --- WAN defaults ---
_wan_defaults = {
    80:  (1280, 720,  81),
    40:  (832,  480,  81),
    24:  (624,  352,  49),
    16:  (480,  272,  33),
}

def _pick(table, vram):
    for threshold in sorted(table.keys(), reverse=True):
        if vram >= threshold:
            return table[threshold]
    return list(table.values())[-1]  # smallest

if MODEL.startswith("ltx"):
    dw, dh, df = _pick(_ltx_defaults, vram)
else:
    dw, dh, df = _pick(_wan_defaults, vram)

# Apply user overrides
_width      = WIDTH      if WIDTH      is not None else dw
_height     = HEIGHT     if HEIGHT     is not None else dh
_num_frames = NUM_FRAMES if NUM_FRAMES is not None else df

# LTX requires num_frames = 8k+1 (1, 9, 17, … 121 …)
if MODEL.startswith("ltx"):
    _num_frames = max(1, ((_num_frames - 1) // 8) * 8 + 1)

duration = _num_frames / FPS
print(f"Resolution : {_width}×{_height}")
print(f"Frames     : {_num_frames} → ~{duration:.1f}s at {FPS} fps")
print(f"CPU offload: {CPU_OFFLOAD}")

## 5. Load the model

First run downloads weights from HuggingFace (~5-25 GB depending on model). Subsequent runs reuse the cache.

> **WAN 14B note:** this model requires ~70+ GB VRAM. It will OOM on anything smaller than an A100 80GB.

In [ ]:
import torch
from diffusers import (
    LTXPipeline,
    LTXImageToVideoPipeline,
    WanPipeline,
)
from diffusers.utils import load_image

dtype = torch.bfloat16  # bfloat16 is more stable than float16 for video models

# ── HuggingFace model IDs ──────────────────────────────────────────────────
_HF_IDS = {
    "ltx-2.3":      "Lightricks/LTX-Video",
    "ltx-2.3-i2v":  "Lightricks/LTX-Video",           # same weights, different pipeline
    "wan-2.1-1.3b": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
    "wan-2.2-a14b":  "Wan-AI/Wan2.2-T2V-A14B-Diffusers",  # MoE: 14B active / 27B total
}

hf_id = _HF_IDS.get(MODEL)
if hf_id is None:
    raise ValueError(f"Unknown MODEL={MODEL!r}. Pick from: {list(_HF_IDS)}")

print(f"Loading {hf_id} …")

if MODEL == "ltx-2.3":
    pipe = LTXPipeline.from_pretrained(hf_id, torch_dtype=dtype)

elif MODEL == "ltx-2.3-i2v":
    pipe = LTXImageToVideoPipeline.from_pretrained(hf_id, torch_dtype=dtype)

elif MODEL in ("wan-2.1-1.3b", "wan-2.2-a14b"):
    pipe = WanPipeline.from_pretrained(hf_id, torch_dtype=dtype)

# ── VRAM optimisations ─────────────────────────────────────────────────────
if CPU_OFFLOAD:
    pipe.enable_model_cpu_offload()
    print("CPU offload enabled")
else:
    pipe.to("cuda")

# VAE slicing + tiling reduces peak VRAM during decoding at no quality cost
if hasattr(pipe, "vae"):
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()

print("Model loaded ✓")

## 6. Generate the video

This is the slow step — expect **3–15 minutes** depending on GPU, resolution, and step count.
Watch VRAM with `!nvidia-smi` in a separate cell if you're worried about OOM.

In [ ]:
import time

generator = torch.Generator(device="cuda").manual_seed(SEED)

common_kwargs = dict(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=_width,
    height=_height,
    num_frames=_num_frames,
    num_inference_steps=NUM_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
)

print(f"Generating {_num_frames} frames at {_width}×{_height} …")
t0 = time.time()

if MODEL == "ltx-2.3-i2v":
    image = load_image(IMAGE_PATH)
    image = image.resize((_width, _height))
    output = pipe(image=image, **common_kwargs)
else:
    output = pipe(**common_kwargs)

elapsed = time.time() - t0
print(f"Done in {elapsed/60:.1f} min")

frames = output.frames[0]   # list of PIL Images
print(f"Got {len(frames)} frames")

## 7. Save & display the video

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import HTML, display
from base64 import b64encode

export_to_video(frames, OUTPUT_PATH, fps=FPS)
print(f"Saved: {OUTPUT_PATH}")

# Display inline in the notebook
with open(OUTPUT_PATH, "rb") as f:
    mp4_b64 = b64encode(f.read()).decode()

display(HTML(
    f'<video controls style="max-width:100%">'
    f'<source src="data:video/mp4;base64,{mp4_b64}" type="video/mp4">'
    f'</video>'
))

In [ ]:
# Download the file to your local machine
from google.colab import files
files.download(OUTPUT_PATH)

---
## Tips & Troubleshooting

### OOM (out of memory)
- Set `CPU_OFFLOAD = True` (already on by default)
- Reduce `NUM_FRAMES` (fewer frames = less VRAM during decoding)
- Reduce `WIDTH` / `HEIGHT`
- Upgrade to a bigger GPU: Runtime → Change runtime type
- Run `torch.cuda.empty_cache()` between attempts

### Bad quality / weird motion
- Increase `NUM_STEPS` to 50-75
- Be more descriptive in your prompt — video models respond well to camera descriptions:
  *"slow dolly shot", "static camera", "aerial view", "close-up"*
- Try different `SEED` values (just change the number and re-run cells 6-7)
- Lower `GUIDANCE_SCALE` slightly (2.5–4.0 is a good range)

### Want image-to-video?
- Set `MODEL = "ltx-2.3-i2v"`
- Upload your image via the Files panel (left sidebar in Colab)
- Set `IMAGE_PATH` to the uploaded file path

### Check GPU while generating
```python
!nvidia-smi
```

### Model card / license links
- LTX-Video: https://huggingface.co/Lightricks/LTX-Video
- WAN 2.2: https://huggingface.co/Wan-AI
- diffusers video docs: https://huggingface.co/docs/diffusers/using-diffusers/video_generation

---
## Optional: generate multiple variations

Run the cell below to sweep over a list of prompts or seeds without reloading the model.

In [ ]:
# Optional batch — tweak prompts/seeds and run this cell
BATCH = [
    {"seed": 42,  "prompt": "A golden retriever running through a field of sunflowers, slow motion, cinematic"},
    {"seed": 123, "prompt": "Time-lapse of clouds moving over a city skyline at sunset, 4K"},
    {"seed": 7,   "prompt": "Underwater coral reef with colorful fish swimming, natural lighting"},
]

for i, item in enumerate(BATCH):
    print(f"\n[{i+1}/{len(BATCH)}] seed={item['seed']}  prompt={item['prompt'][:60]}…")
    gen = torch.Generator(device="cuda").manual_seed(item["seed"])
    out = pipe(
        prompt=item["prompt"],
        negative_prompt=NEGATIVE_PROMPT,
        width=_width,
        height=_height,
        num_frames=_num_frames,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=gen,
    )
    path = f"/content/batch_{i+1:02d}_seed{item['seed']}.mp4"
    export_to_video(out.frames[0], path, fps=FPS)
    print(f"  → saved {path}")

print("\nAll done. Download from the Files panel.")